In [1]:
import re
import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from sentence_transformers import SentenceTransformer

In [2]:
DATA_PATH = 'Data/final_coffee_reviews_absa.csv'
RANDOM_STATE = 42
TOP_K = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
df = pd.read_csv(DATA_PATH)

In [4]:
if "combined_text" not in df.columns:
    raise ValueError("ABSA belum ada. Run preprocessing dulu.")

In [5]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["combined_text"] = df["combined_text"].fillna("").apply(clean_text)

In [6]:
df["origin_country"] = df["Country"].astype(str).str.strip()

In [7]:
df = df[(df["combined_text"].str.len() > 5) & (df["origin_country"] != "")]

# Remove classes with fewer than 2 samples (can't stratify-split otherwise)
counts = df["origin_country"].value_counts()
valid = counts[counts >= 2].index
df = df[df["origin_country"].isin(valid)]

df = df.reset_index(drop=True)
print(f"Rows: {len(df)}, Classes: {df['origin_country'].nunique()}")
print(df["origin_country"].value_counts().tail(5))


Rows: 7577, Classes: 41
origin_country
Malaysia          5
United Kingdom    4
Laos              2
Nepal             2
South Africa      2
Name: count, dtype: int64


In [8]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["origin_country"]
)

In [9]:
def precision_at_k(relevance, k):
    return np.sum(relevance[:k]) / k

def recall_at_k(relevance, total_relevant, k):
    if total_relevant == 0:
        return 0
    return np.sum(relevance[:k]) / total_relevant

def dcg_at_k(relevance, k):
    relevance = np.array(relevance[:k])
    return np.sum(relevance / np.log2(np.arange(2, len(relevance)+2)))

def ndcg_at_k(relevance, total_relevant, k):
    ideal = dcg_at_k([1]*min(total_relevant, k), k)
    if ideal == 0:
        return 0
    return dcg_at_k(relevance, k) / ideal

def average_precision_at_k(relevance, total_relevant, k):
    score = 0
    hits = 0
    for i in range(k):
        if relevance[i]:
            hits += 1
            score += hits / (i+1)
    return score / min(total_relevant, k) if total_relevant > 0 else 0

In [10]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
train_vec = tfidf.fit_transform(train_df["combined_text"])
test_vec = tfidf.transform(test_df["combined_text"])

In [11]:
def evaluate(query_vecs, query_labels, corpus_vecs, corpus_labels):
    sim = cosine_similarity(query_vecs, corpus_vecs)

    results = []

    for i in range(len(query_labels)):
        sims = sim[i]
        idx = np.argsort(-sims)

        ranked_labels = corpus_labels.iloc[idx].values
        true_label = query_labels.iloc[i]

        relevance = [1 if l == true_label else 0 for l in ranked_labels[:TOP_K]]
        total_relevant = np.sum(corpus_labels == true_label)

        results.append({
            "precision": precision_at_k(relevance, TOP_K),
            "recall": recall_at_k(relevance, total_relevant, TOP_K),
            "ndcg": ndcg_at_k(relevance, total_relevant, TOP_K),
            "map": average_precision_at_k(relevance, total_relevant, TOP_K)
        })

    return pd.DataFrame(results).mean()

metrics = evaluate(
    test_vec,
    test_df["origin_country"],
    train_vec,
    train_df["origin_country"]
)

print("=== TF-IDF RESULTS ===")
print(metrics)

=== TF-IDF RESULTS ===
precision    0.168668
recall       0.002858
ndcg         0.175958
map          0.090984
dtype: float64


In [12]:
try:
    model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

    train_emb = model.encode(train_df["combined_text"].tolist(), convert_to_numpy=True)
    test_emb = model.encode(test_df["combined_text"].tolist(), convert_to_numpy=True)

    metrics_sbert = evaluate(
        test_emb,
        test_df["origin_country"],
        train_emb,
        train_df["origin_country"]
    )

    print("\n=== SBERT RESULTS ===")
    print(metrics_sbert)

except Exception as e:
    print("\nSBERT skipped:", e)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== SBERT RESULTS ===
precision    0.165172
recall       0.002705
ndcg         0.170470
map          0.088605
dtype: float64


In [13]:
def recommend_coffee(query, top_n=5):
    if 'model' not in globals() or 'train_emb' not in globals():
        raise ValueError("SBERT model belum siap. Jalankan cell SBERT dulu.")

    query = clean_text(query)

    q_emb = model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(q_emb, train_emb)[0]

    idx = np.argsort(-sims)[:top_n]

    result = train_df.iloc[idx][[
        "Coffee Name",
        "Roaster",
        "origin_country",
        "Blind Assessment"
    ]].copy()

    result.insert(0, "similarity", sims[idx])

    return result.reset_index(drop=True)


In [14]:
query = "chocolate caramel sweet smooth body"
print("\n=== SBERT RECOMMENDATION ===")
print(recommend_coffee(query, 5))


=== SBERT RECOMMENDATION ===
   similarity                                    Coffee Name  \
0    0.685464  Taiwan Nantou Yuchi Kanon Estate SL34 Natural   
1    0.676131                               Ethiopia Celinga   
2    0.660858                      Chuck Roast Guatemala SHB   
3    0.658227           Honduras Catracha “Alfonso” Microlot   
4    0.652346                  Jamaica Blue Mountain (K-Cup)   

                         Roaster origin_country  \
0                  Kakalove Cafe         Taiwan   
1  Greater Goods Coffee Roasters       Ethiopia   
2                   Jones Coffee      Guatemala   
3      Roast Co. Artisan Coffees       Honduras   
4          Green Mountain Coffee        Jamaica   

                                    Blind Assessment  
0  richly sweet-savory. salted caramel, tamarind,...  
1  sweet-toned, spicy. jasmine, caramel, hazelnut...  
2  balanced and quietly complete. sweetly and gen...  
3  lovely balance; complete. caramelly chocolate,...  
4  